# Sistema de Scoring Crediticio con Optimización de Rentabilidad Ajustada por Riesgo

## Contexto del Problema

El riesgo crediticio representa uno de los principales desafíos para las entidades financieras y plataformas de préstamos. Cada vez que una institución otorga un crédito, asume la posibilidad de que el cliente no cumpla con sus obligaciones de pago, generando pérdidas financieras y afectando la rentabilidad del portafolio.

Este proyecto utiliza datos históricos de LendingClub, una plataforma estadounidense de *peer-to-peer lending (P2P lending)* que conectaba inversionistas con personas que solicitaban préstamos personales. Antes de aprobar un préstamo, LendingClub evaluaba el perfil financiero y crediticio de cada solicitante con el objetivo de estimar el riesgo asociado a la operación.

El dataset contiene información sobre millones de préstamos otorgados entre 2007 y 2018, incluyendo variables relacionadas con:

- Perfil financiero del solicitante
- Historial crediticio
- Condiciones del préstamo
- Estado final del crédito

La variable objetivo principal es `loan_status`, que permite identificar si un préstamo fue pagado correctamente o terminó en incumplimiento (*default*).

---

# Objetivo del Proyecto

El objetivo general de este proyecto es desarrollar un sistema de riesgo crediticio basado en Machine Learning capaz de estimar la probabilidad de incumplimiento de un préstamo utilizando información disponible al momento de la solicitud.

A partir de estas predicciones, el proyecto busca simular estrategias de aprobación y analizar cómo distintas políticas de riesgo afectan la rentabilidad y pérdida esperada de un portafolio de créditos.

---

# Objetivos Técnicos

Los objetivos técnicos del proyecto incluyen:

- Realizar un análisis exploratorio de datos (EDA) para comprender la estructura y calidad del dataset.
- Identificar y tratar valores faltantes, variables irrelevantes y posibles problemas de *data leakage*.
- Construir pipelines de preprocesamiento para variables numéricas y categóricas.
- Entrenar y comparar modelos de clasificación para predicción de default.
- Evaluar el desempeño utilizando métricas apropiadas para problemas de riesgo crediticio, como ROC-AUC, Recall y F1-score.
- Interpretar el comportamiento del modelo mediante técnicas de importancia de variables e interpretabilidad.

---

# Objetivos Aplicados al Negocio

Desde una perspectiva financiera, el proyecto busca transformar las probabilidades generadas por el modelo en herramientas de apoyo para la toma de decisiones.

Esto incluye:

- Estimar la probabilidad de default de cada solicitante.
- Construir un sistema de *credit scoring* basado en riesgo.
- Calcular la pérdida esperada (*Expected Loss*) de los préstamos.
- Simular políticas de aprobación utilizando distintos umbrales de riesgo.
- Optimizar decisiones de otorgamiento de crédito para maximizar rentabilidad y minimizar pérdidas.

De esta manera, el proyecto no se limita únicamente a un problema de clasificación, sino que incorpora conceptos reales de modelado de riesgo financiero y optimización de decisiones.

## 1.0 Librerías

In [202]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
pd.set_option('display.max_rows', None)


pd.set_option('display.max_columns', None)

In [203]:
df = pd.read_csv(r"C:\Users\camil\Documents\Laburo\Portafolio\credit-risk\data\raw\accepted_2007_to_2018Q4.csv", low_memory= False)

In [204]:
dict_df = pd.read_excel(r"C:\Users\camil\Documents\Laburo\Portafolio\credit-risk\data\raw\LCDataDictionary.xlsx")

c:\Users\camil\anaconda3\envs\mlp311\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [205]:
browse_feat = dict_df["LoanStatNew"].dropna().values

In [206]:
browse_feat = [x.replace("\xa0", "") for x in browse_feat]

correcciones = {
    "is_inc_v": "verification_status",
    "verified_status_joint": "verification_status_joint"
}

browse_feat = [correcciones.get(col, col) for col in browse_feat]

In [207]:
valid_features = list(set(browse_feat).intersection(set(df.columns)))

In [208]:
len(valid_features)

76

In [209]:
df_browse = df[valid_features].copy()
df_browse["loan_status"] = df["loan_status"]

## 1.1 Entendimiento de los datos

In [210]:
df_browse.head()

,dti_joint,open_acc_6m,desc,collection_recovery_fee,total_bal_il,zip_code,funded_amnt,earliest_cr_line,fico_range_low,loan_amnt,addr_state,total_rec_int,annual_inc_joint,tot_cur_bal,next_pymnt_d,funded_amnt_inv,issue_d,id,recoveries,out_prncp_inv,revol_util,mths_since_rcnt_il,purpose,grade,tot_coll_amt,pub_rec,fico_range_high,annual_inc,inq_last_12m,inq_last_6mths,home_ownership,mths_since_last_delinq,last_pymnt_d,revol_bal,il_util,total_pymnt,verification_status_joint,acc_now_delinq,open_rv_12m,last_fico_range_high,dti,delinq_2yrs,mths_since_last_record,open_acc,policy_code,collections_12_mths_ex_med,last_fico_range_low,initial_list_status,application_type,emp_title,loan_status,term,sub_grade,int_rate,emp_length,open_il_12m,inq_fi,last_credit_pull_d,max_bal_bc,total_cu_tl,out_prncp,all_util,url,total_acc,open_rv_24m,pymnt_plan,last_pymnt_amnt,open_il_24m,total_rec_prncp,installment,title,member_id,total_pymnt_inv,verification_status,total_rec_late_fee,mths_since_last_major_derog
0,NaN,2.0,NaN,0.0,4981.0,190xx,3600.0,Aug-2003,675.0,3600.0,PA,821.72,NaN,144904.0,NaN,3600.0,Dec-2015,68407277,0.0,0.00,29.7,21.0,debt_consolidation,C,722.0,0.0,679.0,55000.0,4.0,1.0,MORTGAGE,30.0,Jan-2019,2765.0,36.0,4421.723917,NaN,0.0,3.0,564.0,5.91,0.0,NaN,7.0,1.0,0.0,560.0,w,Individual,leadman,Fully Paid,36 months,C4,13.99,10+ years,0.0,3.0,Mar-2019,722.0,1.0,0.00,34.0,https://lendingclub.com/browse/loanDetail.acti...,13.0,3.0,n,122.67,1.0,3600.00,123.03,Debt consolidation,NaN,4421.72,Not Verified,0.0,30.0
1,NaN,1.0,NaN,0.0,18005.0,577xx,24700.0,Dec-1999,715.0,24700.0,SD,979.66,NaN,204396.0,NaN,24700.0,Dec-2015,68355089,0.0,0.00,19.2,19.0,small_business,C,0.0,0.0,719.0,65000.0,6.0,4.0,MORTGAGE,6.0,Jun-2016,21470.0,73.0,25679.660000,NaN,0.0,2.0,699.0,16.06,1.0,NaN,22.0,1.0,0.0,695.0,w,Individual,Engineer,Fully Paid,36 months,C1,11.99,10+ years,0.0,0.0,Mar-2019,6472.0,0.0,0.00,29.0,https://lendingclub.com/browse/loanDetail.acti...,38.0,3.0,n,926.35,1.0,24700.00,820.28,Business,NaN,25679.66,Not Verified,0.0,NaN
2,13.85,0.0,NaN,0.0,10827.0,605xx,20000.0,Aug-2000,695.0,20000.0,IL,2705.92,71000.0,189699.0,NaN,20000.0,Dec-2015,68341763,0.0,0.00,56.2,19.0,home_improvement,B,0.0,0.0,699.0,63000.0,1.0,0.0,MORTGAGE,NaN,Jun-2017,7869.0,73.0,22705.924294,Not Verified,0.0,0.0,704.0,10.78,0.0,NaN,6.0,1.0,0.0,700.0,w,Joint App,truck driver,Fully Paid,60 months,B4,10.78,10+ years,0.0,2.0,Mar-2019,2081.0,5.0,0.00,65.0,https://lendingclub.com/browse/loanDetail.acti...,18.0,2.0,n,15813.30,4.0,20000.00,432.66,NaN,NaN,22705.92,Not Verified,0.0,NaN
3,NaN,1.0,NaN,0.0,12609.0,076xx,35000.0,Sep-2008,785.0,35000.0,NJ,12361.66,NaN,301500.0,Apr-2019,35000.0,Dec-2015,66310712,0.0,15897.65,11.6,23.0,debt_consolidation,C,0.0,0.0,789.0,110000.0,0.0,0.0,MORTGAGE,NaN,Feb-2019,7802.0,70.0,31464.010000,NaN,0.0,1.0,679.0,17.06,0.0,NaN,13.0,1.0,0.0,675.0,w,Individual,Information Systems Officer,Current,60 months,C5,14.85,10+ years,0.0,0.0,Mar-2019,6987.0,1.0,15897.65,45.0,https://lendingclub.com/browse/loanDetail.acti...,17.0,1.0,n,829.90,1.0,19102.35,829.90,Debt consolidation,NaN,31464.01,Source Verified,0.0,NaN
4,NaN,1.0,NaN,0.0,73839.0,174xx,10400.0,Jun-1998,695.0,10400.0,PA,1340.50,NaN,331730.0,NaN,10400.0,Dec-2015,68476807,0.0,0.00,64.5,14.0,major_purchase,F,0.0,0.0,699.0,104433.0,3.0,3.0,MORTGAGE,12.0,Jul-2016,21929.0,84.0,11740.500000,NaN,0.0,4.0,704.0,25.37,1.0,NaN,12.0,1.0,0.0,700.0,w,Individual,Contract Specialist,Fully Paid,60 months,F1,22.45,3 years,0.0,2.0,Mar-2018,9702.0,1.0,0.00,78.0,https://lendingclub.com/browse/loanDetail.acti...,35.0,7.0,n,10128.96,3.0,10400.00,289.91,Major purchase,NaN,11740.50,Source Verified,0.0,NaN


In [211]:
print(f'The dataset has {df_browse.shape[0]} rows and {df_browse.shape[1]} columns.')

The dataset has 2260701 rows and 76 columns.


## 1.2 Selección y Comprensión de Variables

El dataset original de LendingClub contiene más de 150 variables. Sin embargo, muchas de ellas corresponden a información generada después de que el préstamo fue aprobado o durante su ciclo de vida. Estas variables no estarían disponibles en el momento real de evaluar a un solicitante y, por lo tanto, introducirían problemas de *data leakage* si fueran utilizadas para entrenar el modelo.

Para identificar las variables realmente disponibles para inversionistas y analistas en el momento de la solicitud del préstamo, se utilizó el diccionario oficial de datos de LendingClub (`LCDataDictionary.xlsx`), específicamente la hoja denominada *Browse Notes*, publicada por Wendy Kan.

A partir de esta referencia, se realizó una intersección entre:

- Las variables presentes en el dataset original
- Las variables documentadas como visibles para inversionistas en el diccionario de datos

Este proceso permitió reducir el conjunto inicial de variables y conservar únicamente aquellas relevantes para un escenario realista de predicción de riesgo crediticio.

Posteriormente, durante las etapas de análisis exploratorio y modelado, algunas variables adicionales serán eliminadas debido a:

- Altos porcentajes de valores faltantes
- Baja utilidad predictiva
- Cardinalidad excesiva
- Posibles problemas de fuga de información (*data leakage*)

No obstante, ciertas variables serán conservadas temporalmente durante el EDA con el fin de entender mejor el comportamiento financiero de los préstamos, incluso si posteriormente no son utilizadas para entrenar el modelo final.


## 1.2.1 Diccionario de Variables — Lending Club

A continuación se presenta la descripción técnica y funcional de las variables utilizadas en el proyecto de riesgo crediticio con Lending Club. Las variables se agrupan según su naturaleza y utilidad dentro del análisis exploratorio, modelado predictivo y evaluación financiera.

---

### 1. Variables de Identificación y Metadatos

| Variable | Descripción |
|---|---|
| `id` | Identificador único del préstamo. | 
| `member_id` | Identificador único del cliente en Lending Club. |
| `url` | URL asociada al préstamo dentro de Lending Club. | 

---

### 2. Información del Préstamo y Condiciones Financieras

| Variable | Descripción |
|---|---|
| `loan_amnt` | Monto total solicitado por el prestatario. |
| `funded_amnt` | Monto financiado por Lending Club. |
| `funded_amnt_inv` | Monto financiado por inversionistas. |
| `term` | Duración del préstamo (36 o 60 meses). |
| `int_rate` | Tasa de interés anual aplicada al préstamo. |
| `installment` | Cuota mensual que debe pagar el prestatario. |
| `grade` | Calificación crediticia general asignada por Lending Club. |
| `sub_grade` | Subcategoría detallada de riesgo dentro del grade. |
| `purpose` | Motivo declarado del préstamo. |
| `title` | Título descriptivo asignado por el prestatario. |
| `issue_d` | Fecha de emisión del préstamo. |
| `application_type` | Tipo de aplicación: individual o conjunta (*joint application*). |
| `initial_list_status` | Estado inicial del préstamo en el marketplace (Whole/Fractional). |

---

### 3. Información Laboral y Socioeconómica

| Variable | Descripción |
|---|---|
| `emp_title` | Cargo laboral reportado por el prestatario. |
| `emp_length` | Antigüedad laboral en años. |
| `home_ownership` | Tipo de tenencia de vivienda (RENT, OWN, MORTGAGE). |
| `annual_inc` | Ingreso anual reportado por el prestatario. |
| `verification_status` | Estado de verificación de ingresos. |
| `annual_inc_joint` | Ingreso anual combinado para aplicaciones conjuntas. |
| `verification_status_joint` | Estado de verificación de ingresos en aplicaciones conjuntas. |
| `zip_code` | Código postal parcial del prestatario. |
| `addr_state` | Estado de residencia del prestatario. |

---

### 4. Variables de Riesgo Crediticio e Historial Financiero

| Variable | Descripción |
|---|---|
| `dti` | Relación deuda-ingreso (*Debt-to-Income Ratio*). |
| `dti_joint` | Relación deuda-ingreso para aplicaciones conjuntas. |
| `fico_range_low` | Límite inferior del rango FICO del prestatario. |
| `fico_range_high` | Límite superior del rango FICO del prestatario. |
| `last_fico_range_low` | Último rango FICO inferior registrado. |
| `last_fico_range_high` | Último rango FICO superior registrado. |
| `delinq_2yrs` | Número de moras en los últimos 2 años. |
| `acc_now_delinq` | Número de cuentas actualmente en mora. |
| `pub_rec` | Número de registros públicos negativos (ej. bancarrota). |
| `open_acc` | Número de cuentas de crédito abiertas. |
| `total_acc` | Número total de cuentas de crédito en historial. |
| `revol_bal` | Balance total de crédito revolvente utilizado. |
| `revol_util` | Utilización del crédito revolvente (%). |
| `inq_last_6mths` | Número de consultas de crédito en los últimos 6 meses. |
| `inq_last_12m` | Número de consultas de crédito en los últimos 12 meses. |
| `earliest_cr_line` | Fecha de apertura de la línea de crédito más antigua. |
| `mths_since_last_delinq` | Meses desde la última mora registrada. |
| `mths_since_last_record` | Meses desde el último registro público negativo. |
| `mths_since_last_major_derog` | Meses desde el último evento derogatorio severo. |

---

### 5. Variables Derivadas de Crédito e Información Bancaria

| Variable | Descripción |
|---|---|
| `tot_cur_bal` | Balance total actual de todas las cuentas. |
| `tot_coll_amt` | Total de cuentas enviadas a cobranza. |
| `total_rev_hi_lim` | Límite total de crédito revolvente disponible. |
| `total_bal_il` | Balance total de líneas de crédito a plazos (*installment loans*). |
| `all_util` | Utilización total del crédito disponible. |
| `il_util` | Utilización de líneas de crédito a plazos. |
| `max_bal_bc` | Balance máximo en tarjetas bancarias. |
| `total_cu_tl` | Número total de líneas de crédito financiero abiertas. |

---

### 6. Variables Relacionadas con Apertura y Actividad de Crédito

| Variable | Descripción |
|---|---|
| `open_acc_6m` | Número de cuentas abiertas en los últimos 6 meses. |
| `open_il_12m` | Número de líneas de crédito a plazos abiertas en 12 meses. |
| `open_il_24m` | Número de líneas de crédito a plazos abiertas en 24 meses. |
| `open_rv_12m` | Número de líneas revolventes abiertas en 12 meses. |
| `open_rv_24m` | Número de líneas revolventes abiertas en 24 meses. |
| `open_il_6m` | Número de líneas de crédito a plazos abiertas en 6 meses. |
| `mths_since_rcnt_il` | Meses desde la apertura más reciente de una línea a plazos. |
| `inq_fi` | Número de consultas financieras personales recientes. |

---

### 7. Estado del Préstamo y Variables Post-Emisión (Potential Data Leakage)

Estas variables contienen información generada después de la aprobación del préstamo y, aunque son útiles para análisis financieros y EDA, no deben utilizarse para entrenar modelos predictivos de default.

| Variable | Descripción |
|---|---|
| `loan_status` | Estado final o actual del préstamo. |
| `pymnt_plan` | Indica si existe un plan de pagos especial. |
| `out_prncp` | Principal pendiente por pagar. |
| `out_prncp_inv` | Principal pendiente correspondiente a inversionistas. |
| `total_pymnt` | Total pagado por el prestatario. |
| `total_pymnt_inv` | Total pagado a inversionistas. |
| `total_rec_prncp` | Capital recuperado. |
| `total_rec_int` | Intereses recuperados. |
| `total_rec_late_fee` | Cargos por mora recuperados. |
| `recoveries` | Monto recuperado después de default. |
| `collection_recovery_fee` | Comisiones asociadas a recuperación de deuda. |
| `last_pymnt_d` | Fecha del último pago realizado. |
| `last_pymnt_amnt` | Monto del último pago realizado. |
| `next_pymnt_d` | Próxima fecha estimada de pago. |
| `last_credit_pull_d` | Última fecha de consulta crediticia realizada por Lending Club. |

---

### 8. Variables Especiales para Aplicaciones Conjuntas (Joint Applications)

Estas variables solo aplican a préstamos realizados por más de un prestatario y presentan altos porcentajes de valores faltantes debido a que la mayoría de préstamos son individuales.

| Variable | Descripción |
|---|---|
| `annual_inc_joint` | Ingreso anual combinado de solicitantes conjuntos. |
| `dti_joint` | Relación deuda-ingreso conjunta. |
| `verification_status_joint` | Estado de verificación de ingresos conjuntos. |

---

### Consideraciones Técnicas

- Algunas variables con altos porcentajes de valores faltantes fueron conservadas inicialmente para análisis exploratorio.
- Variables como `grade` y `sub_grade` representan sistemas internos de scoring desarrollados por Lending Club y serán comparadas posteriormente con los modelos de Machine Learning desarrollados en este proyecto.

## 1.3 Información general

In [212]:
df_browse.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Data columns (total 76 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   dti_joint                    float64
 1   open_acc_6m                  float64
 2   desc                         object 
 3   collection_recovery_fee      float64
 4   total_bal_il                 float64
 5   zip_code                     object 
 6   funded_amnt                  float64
 7   earliest_cr_line             object 
 8   fico_range_low               float64
 9   loan_amnt                    float64
 10  addr_state                   object 
 11  total_rec_int                float64
 12  annual_inc_joint             float64
 13  tot_cur_bal                  float64
 14  next_pymnt_d                 object 
 15  funded_amnt_inv              float64
 16  issue_d                      object 
 17  id                           object 
 18  recoveries                   float64
 19  

 - Con un primer acercamiento encontramos qué podemos eliminar variables sin valor predictivo, ni analítico como lo son:
     - `id`
     - `member_id`
     - `url`

- También se identificaron variables con posible *data leakage*.
  - Estas variables contienen información generada después de la emisión del préstamo.
  - Por tanto, no pueden utilizarse en un modelo de originación crediticia (*application credit scoring*).

- Entre las variables con *data leakage* se encuentran:
  - `recoveries`
  - `collection_recovery_fee`
  - `total_rec_prncp`
  - `total_rec_int`
  - `last_pymnt_amnt`
  - `last_pymnt_d`
  - `next_pymnt_d`
  - `out_prncp`
  - `total_pymnt` 
  - `recoveries`

- Estas variables podrían inflar artificialmente el desempeño del modelo y serán excluidas del entrenamiento.


- Algunas variables del dataset se encuentran almacenadas en tipos de datos que no representan correctamente su naturaleza estadística o temporal, por lo que es necesario realizar conversiones antes del análisis y modelado.


- Conversión de variables temporales desde `object` a formato `datetime`, especialmente:
  - `issue_d`
  - `earliest_cr_line`
  - `last_credit_pull_d`

- Conversión de variables categóricas ordinales y estructuradas:
  - `term`: actualmente almacenada como texto (`"36 months"`, `"60 months"`), debe convertirse a formato numérico entero.
  - `emp_length`: contiene categorías textuales (`"10+ years"`, `"2 years"`, etc.) y debe transformarse a una representación ordinal numérica.




In [213]:
# -----------------------------------
# Convert date variables to datetime
# -----------------------------------

date_cols = [
    'issue_d',
    'earliest_cr_line',
    'last_credit_pull_d',
]

# Convert columns
for col in date_cols:
    
    df_browse[col] = pd.to_datetime(
        df_browse[col],
        format='%b-%Y',
        errors='coerce'
    )

# Check result
df_browse[date_cols].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Data columns (total 3 columns):
 #   Column              Dtype         
---  ------              -----         
 0   issue_d             datetime64[ns]
 1   earliest_cr_line    datetime64[ns]
 2   last_credit_pull_d  datetime64[ns]
dtypes: datetime64[ns](3)
memory usage: 51.7 MB


In [214]:
# ============================================
# Eliminación de variables irrelevantes,
# con data leakage y exceso de valores faltantes
# ============================================

cols_to_drop = [
    # ------------------------
    # Variables sin valor predictivo
    # ------------------------
    "id",
    "member_id",
    "url",

    # ------------------------
    # Variables con data leakage y redundancia
    # ------------------------
    "recoveries",
    "collection_recovery_fee",
    "total_rec_prncp",
    "total_rec_int",
    "last_pymnt_amnt",
    "last_pymnt_d",
    "next_pymnt_d",
    'out_prncp',
    'out_prncp_inv',
    'total_pymnt',
    'total_pymnt_inv',
    'total_rec_late_fee',
    'last_fico_range_high',
    'last_fico_range_low',
    'last_credit_pull_d',
     'funded_amnt_inv',
]





# Eliminación de columnas
df_browse = df_browse.drop(columns=cols_to_drop, errors="ignore")

print(f"Dataset después de limpieza inicial: {df_browse.shape}")

Dataset después de limpieza inicial: (2260701, 57)


## 1.4 Variables numéricas y catégoricas

In [215]:
num_cols = df_browse.select_dtypes(include='number').columns.tolist()
cat_cols = df_browse.select_dtypes(include='object').columns.tolist()
date_cols = [c for c in df_browse.columns if 'd' in c]

print(f'Hay un total de {len(num_cols)} variables numericas en el dataset.')
print(f'Hay un total de  {len(cat_cols)} variables categoricas en el dataset .')
print(f'Hay un total de  {len(date_cols)} variables temporales en el dataset .')

Hay un total de 38 variables numericas en el dataset.
Hay un total de  17 variables categoricas en el dataset .
Hay un total de  16 variables temporales en el dataset .


In [216]:
cat_summary = pd.DataFrame({
    'variable': cat_cols,
    'cardinalidad': [df_browse[col].nunique() for col in cat_cols],
})

cat_summary = cat_summary.sort_values(
    by='cardinalidad',
    ascending=False
)

cat_summary

,variable,cardinalidad
9,emp_title,512694
0,desc,124500
15,title,63154
1,zip_code,956
2,addr_state,51
12,sub_grade,35
3,purpose,14
13,emp_length,11
10,loan_status,9
4,grade,7


In [217]:
import pandas as pd

# Seleccionar variables categóricas

summary = []

for col in cat_cols:
    
    vc = df_browse[col].value_counts(normalize=True, dropna=False)
    
    summary.append({
        'variable': col,
        'top_category': vc.index[0],
        'top_category_%': round(vc.iloc[0] * 100, 2),
        'second_category_%': round(vc.iloc[1] * 100, 2) if len(vc) > 1 else 0,
        'is_constant': vc.iloc[0] >= 0.999,
        'highly_imbalanced': vc.iloc[0] >= 0.95,
    })

cat_balance_summary = pd.DataFrame(summary)

cat_balance_summary = cat_balance_summary.sort_values(
    by='top_category_%',
    ascending=False
)

cat_balance_summary

,variable,top_category,top_category_%,second_category_%,is_constant,highly_imbalanced
14,pymnt_plan,n,99.97,0.03,True,True
6,verification_status_joint,NaN,94.88,2.54,False,False
8,application_type,Individual,94.66,5.34,False,False
0,desc,NaN,94.42,0.01,False,False
11,term,36 months,71.21,28.79,False,False
7,initial_list_status,w,67.92,32.08,False,False
3,purpose,debt_consolidation,56.53,22.87,False,False
15,title,Debt consolidation,51.01,20.78,False,False
5,home_ownership,MORTGAGE,49.16,39.59,False,False
10,loan_status,Fully Paid,47.63,38.85,False,False


- Por cardinalidad excesiva se eliminaran las variables:
    - `emp_title`,
    - `title`,
    - `zip_code`
- Por desbalance excesivo o casi constancia de categorías serán eliminadas:
    - `pymnt_plan`

In [218]:
cols_to_drop = [
    # ------------------------
    # Variables con cardinalidad excesiva
    # ------------------------
    "emp_title",
    "title",
    "zip_code",

    # ------------------------
    # Variables constantes o excesivamente desbalanceadas
    # ------------------------

    "pymnt_plan"
]

# Eliminación de columnas
df_browse = df_browse.drop(columns=cols_to_drop, errors="ignore")

print(f"Dataset después de limpieza inicial: {df_browse.shape}")

Dataset después de limpieza inicial: (2260701, 53)


## 1.5 Análisis de datos faltantes y duplicados

In [219]:
missing_table = pd.DataFrame({
    "Valores_Faltantes": df_browse.isna().sum(),
    "Porcentaje": df_browse.isna().mean() * 100,
})

missing_table = missing_table.sort_values(by="Porcentaje", ascending=False)
missing_table

,Valores_Faltantes,Porcentaje
verification_status_joint,2144971,94.880791
dti_joint,2139995,94.660683
annual_inc_joint,2139991,94.660506
desc,2134636,94.423632
mths_since_last_record,1901545,84.113069
mths_since_last_major_derog,1679926,74.309960
mths_since_last_delinq,1158535,51.246715
il_util,1068883,47.281042
mths_since_rcnt_il,909957,40.251099
all_util,866381,38.323555


In [220]:
df_browse.duplicated().sum()

np.int64(32)

- El dataset presenta 32 filas duplicadas que serán investigadas en el siguiente paso

- Se establecerá un umbral de eliminación del 70% de valores faltantes.
  - Las variables con un porcentaje igual o superior a este umbral serán eliminadas del análisis principal.
  - Esto se debe a que:
    - Su capacidad informativa es limitada.
    - La imputación podría introducir sesgos importantes.
    - Algunas corresponden a casos muy específicos o poco representativos.
    - Otras contienen información posterior a la aprobación del préstamo (*data leakage*).

  - Bajo este criterio, variables como:
    - `member_id`
    - `verification_status_joint`
    - `dti_joint`
    - `annual_inc_joint`
    - `desc`
    - `mths_since_last_record`
    - `mths_since_last_major_derog`
  
  serán excluidas del análisis principal.

- La presencia de valores faltantes puede deberse a distintas razones:
  - El solicitante no proporcionó la información.
  - La información no fue registrada correctamente.
  - El valor no aplicaba para ciertos clientes.
  - El valor faltante realmente representa ausencia de eventos financieros.
  - Cambios históricos en la captura de variables dentro de LendingClub.

- Dependiendo del comportamiento de los valores faltantes, se aplicarán distintas estrategias:
  - Eliminación de variables con:
    - porcentaje excesivo de nulos,
    - alta cardinalidad irrelevante,
    - o ausencia de valor predictivo.
  
  - Eliminación de observaciones:
    - En variables con porcentajes mínimos de valores faltantes (<1%), los registros afectados podrán eliminarse directamente debido al gran tamaño del dataset.
  
  - Imputación:
    - Se considerarán técnicas como:
      - media,
      - mediana,
      - moda,
      - o categorías como `"Unknown"` para variables categóricas.
  
  - Tratamiento del missing como información:
    - En ciertos casos, la ausencia de información puede estar asociada al perfil de riesgo del solicitante.
    - Por ejemplo, no tener historial crediticio podría representar un perfil financiero distinto.
    - En estos escenarios, el valor faltante podrá tratarse como una categoría adicional.
- Existen variables con exactamente el mismo porcentaje de missing, por lo qué probablemente indica un patrón estructural
- Procederemos a hacer una investigación del comportamiendo de los y causa de los duplicados y valores faltantes

In [221]:
cols_to_drop = [
    # ------------------------
    # Variables con más de 70% de missing
    # ------------------------
    "verification_status_joint",
    "dti_joint",
    "annual_inc_joint",
    "desc",
    "mths_since_last_record",
    "mths_since_last_major_derog"
]

# Eliminación de columnas
df_browse = df_browse.drop(columns=cols_to_drop, errors="ignore")

print(f"Dataset después de limpieza inicial: {df_browse.shape}")

Dataset después de limpieza inicial: (2260701, 47)


### 1.5.1 Investigando los NAs y duplicados

### 1.5.1.1 Duplicados

In [222]:
duplicates = df_browse[df_browse.duplicated(keep=False)]

duplicates.sort_values(by="loan_status").head(32)

,open_acc_6m,total_bal_il,funded_amnt,earliest_cr_line,fico_range_low,loan_amnt,addr_state,tot_cur_bal,issue_d,revol_util,mths_since_rcnt_il,purpose,grade,tot_coll_amt,pub_rec,fico_range_high,annual_inc,inq_last_12m,inq_last_6mths,home_ownership,mths_since_last_delinq,revol_bal,il_util,acc_now_delinq,open_rv_12m,dti,delinq_2yrs,open_acc,policy_code,collections_12_mths_ex_med,initial_list_status,application_type,loan_status,term,sub_grade,int_rate,emp_length,open_il_12m,inq_fi,max_bal_bc,total_cu_tl,all_util,total_acc,open_rv_24m,open_il_24m,installment,verification_status
421095,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
421096,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
528961,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
528962,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
651664,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
651665,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
749520,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
749521,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
877716,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
877717,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [223]:
# Remove fully empty and duplicated rows
df_browse = (
    df_browse
    .dropna(how='all')
    .drop_duplicates()
)

### 1.4.1.2 NAs

In [224]:
# -----------------------------------------
# Investigate whether variables were added
# in later years
# -----------------------------------------

vars_to_check = [
    'il_util',
    'mths_since_rcnt_il',
    'all_util',
    'total_cu_tl',
    'open_acc_6m',
    'inq_last_12m',
    'inq_fi',
    'open_rv_12m',
    'max_bal_bc',
    'open_il_12m',
    'total_bal_il',
    'open_rv_24m',
    'open_il_24m',
]

# Create issue year
df_browse['issue_year'] = df_browse['issue_d'].dt.year

# Missing percentage by year
missing_by_year = (
    df_browse
    .groupby('issue_year')[vars_to_check]
    .apply(lambda x: x.isna().mean() * 100)
    .round(2)
)

missing_by_year

,il_util,mths_since_rcnt_il,all_util,total_cu_tl,open_acc_6m,inq_last_12m,inq_fi,open_rv_12m,max_bal_bc,open_il_12m,total_bal_il,open_rv_24m,open_il_24m
issue_year,,,,,,,,,,,,,
2007,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
2008,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
2009,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
2010,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
2011,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
2012,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
2013,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
2014,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
2015,95.58,95.06,94.92,94.92,94.92,94.92,94.92,94.92,94.92,94.92,94.92,94.92,94.92


In [225]:
cols = [
    'il_util','mths_since_rcnt_il'
]

df_browse['issue_year'] = pd.to_datetime(df_browse['issue_d']).dt.year

In [226]:
for c in cols:
    df_browse[c + "_missing"] = df_browse[c].isna().astype(int)

df_browse.groupby('il_util_missing')['loan_status'].value_counts(normalize=True)

il_util_missing  loan_status                                        
0                Current                                                0.587223
                 Fully Paid                                             0.302158
                 Charged Off                                            0.088016
                 Late (31-120 days)                                     0.014214
                 In Grace Period                                        0.005522
                 Late (16-30 days)                                      0.002842
                 Default                                                0.000026
1                Fully Paid                                             0.670472
                 Current                                                0.166959
                 Charged Off                                            0.153118
                 Late (31-120 days)                                     0.004235
                 Does not meet the credi

In [243]:
df_browse['loan_status'].value_counts(normalize=True) * 100

loan_status
Fully Paid                                             47.660592
Current                                                38.818632
Charged Off                                            11.887313
Late (31-120 days)                                      0.948851
In Grace Period                                         0.373010
Late (16-30 days)                                       0.192331
Does not meet the credit policy. Status:Fully Paid      0.084801
Does not meet the credit policy. Status:Charged Off     0.032698
Default                                                 0.001772
Name: proportion, dtype: float64

In [247]:
# Assess missing values loan_status distribution in emp_length to see whether their occurrence is at random.
df_browse.loc[df['emp_length'].isna(), 'loan_status'].value_counts(normalize=True) * 100


loan_status
Current                                                44.208539
Fully Paid                                             39.305052
Charged Off                                            14.499662
Late (31-120 days)                                      1.256527
In Grace Period                                         0.397487
Late (16-30 days)                                       0.305865
Does not meet the credit policy. Status:Fully Paid      0.013089
Does not meet the credit policy. Status:Charged Off     0.010333
Default                                                 0.003444
Name: proportion, dtype: float64

In [244]:
df_browse['mths_since_last_delinq'].describe()

count    1.100446e+06
mean     3.454699e+01
std      2.189993e+01
min      0.000000e+00
25%      1.600000e+01
50%      3.100000e+01
75%      5.000000e+01
max      2.260000e+02
Name: mths_since_last_delinq, dtype: float64

In [230]:
(
    df_browse
    .assign(
        mths_last_delinq_missing=
        df_browse['mths_since_last_delinq'].isna()
    )
    .groupby('mths_last_delinq_missing')['loan_status']
    .value_counts(normalize=True)
)

mths_last_delinq_missing  loan_status                                        
False                     Fully Paid                                             0.479737
                          Current                                                0.377146
                          Charged Off                                            0.125042
                          Late (31-120 days)                                     0.010253
                          In Grace Period                                        0.004210
                          Late (16-30 days)                                      0.002186
                          Does not meet the credit policy. Status:Fully Paid     0.000984
                          Does not meet the credit policy. Status:Charged Off    0.000423
                          Default                                                0.000020
True                      Fully Paid                                             0.473026
                      

- Se identificó que las siguientes variables se incorporaron al dataset a partir del año 2015. Por lo tanto, para mantener la integridad de los datos históricos, se manejarán como variables no disponibles en periodos anteriores y se transformarán sus valores nulos (`NA`) mediante un enfoque combinado de indicador de ausencia y tratamiento numérico:

    - `all_util`
    - `total_cu_tl`
    - `open_acc_6m`
    - `inq_last_12m`
    - `inq_fi`
    - `open_rv_12`
    - `max_bal_bc`
    - `open_il_12m`
    - `total_bal_il`
    - `open_rv_24m`
    - `open_il_24m`
    - `il_util`
    - `mths_since_rcnt_il`

  Para estas variables se aplicará:
  1. Un **indicador de disponibilidad (missingness flag)** para capturar la ausencia de información.
  2. En registros anteriores a 2015, donde la variable no existía, se asignará el valor **-1** como representación de “no aplica”.
  3. En registros posteriores a 2015, los valores nulos se imputarán con la **mediana de la variable**, preservando la distribución sin introducir sesgos extremos.
  4. Nota: Para il_util y mths_since_rcnt_il, además de la ausencia estructural previa a 2015, se observa también missing en periodos posteriores, por lo que el indicador de ausencia captura tanto falta estructural como falta de reporte

- Para las variables que presentan una proporción de valores faltantes inferior al **1%**, se optará por eliminar las filas vacías, ya que su impacto en el volumen total del dataset es insignificante:

    - `revol_util`
    - `dti`
    - `collections_12_mths_ex_med`
    - `last_credit_pull_d`
    - `inq_last_6mths`
    - `open_acc`
    - `earliest_cr_line`
    - `total_acc`
    - `pub_rec`
    - `delinq_2yrs`
    - `acc_now_delinq`

- Para `emp_length`: No hay algún patrón identificable con default y no default repecto al target por lo que los valores faltantes se transformarán en la categoría **"unknown"**, evitando imputaciones numéricas que puedan inducir interpretaciones incorrectas del estado laboral.

- Para `tot_cur_bal` y `tot_coll_amt`: Se aplicará una estrategia doble:
    1. Se creará un **indicador binario de ausencia** (flag de missingness).
    2. Los valores nulos se imputarán con **0**, dado que en contexto financiero un valor ausente suele ser consistente con ausencia de saldo o actividad reportada.
- Para `mths_since_last_delinq`: Los valores faltantes se interpretarán como clientes que probablemente nunca han presentado morosidad (delinquency). En lugar de imputar con media o mediana, se aplicará:

    1. Un indicador binario de missingness.
    2. Una imputación con el valor máximo observado + 1, representando un estado de “nunca en mora” y preservando la estructura ordinal de riesgo de la variable.


In [231]:
# ============================================
# 0. Preparación temporal
# ============================================

df_browse['issue_year'] = df_browse['issue_d'].dt.year

post_mask = df_browse['issue_year'] >= 2015
# ============================================
# 1. Variables estructuralmente inexistentes antes de 2015
# ============================================

pre_2015_vars = [
    'all_util',
    'total_cu_tl',
    'open_acc_6m',
    'inq_last_12m',
    'inq_fi',
    'open_rv_12m',
    'max_bal_bc',
    'open_il_12m',
    'total_bal_il',
    'open_rv_24m',
    'open_il_24m'
]

for col in pre_2015_vars:

    # missing SOLO post-2015
    df_browse[col + '_missing'] = (
        post_mask & df_browse[col].isna()
    ).astype(int)

    # mediana SOLO post-2015 (evita leakage)
    median_post = df_browse.loc[
        post_mask & df_browse[col].notna(),
        col
    ].median()

    # imputación post-2015
    df_browse.loc[post_mask, col] = df_browse.loc[post_mask, col].fillna(median_post)

    # pre-2015 = no existía
    df_browse.loc[~post_mask, col] = -1

In [232]:
# ============================================
# 2. IL_UTIL (caso mixto: missing + estructural)
# ============================================

df_browse['il_util_missing'] = (
    (post_mask) & (df_browse['il_util'].isna())
).astype(int)

df_browse['il_util_not_applicable'] = (df_browse['issue_year'] < 2015).astype(int)

median_il_util_post = df_browse.loc[
    post_mask & df_browse['il_util'].notna(),
    'il_util'
].median()

df_browse.loc[post_mask, 'il_util'] = df_browse.loc[post_mask, 'il_util'].fillna(median_il_util_post)

df_browse.loc[~post_mask, 'il_util'] = -1

In [233]:
# ============================================
# 3. MTHS_SINCE_RCNT_IL (recency variable)
# ============================================

df_browse['mths_since_rcnt_il_missing'] = df_browse['mths_since_rcnt_il'].isna().astype(int)

median_rcnt_il_post = df_browse.loc[
    post_mask & df_browse['mths_since_rcnt_il'].notna(),
    'mths_since_rcnt_il'
].median()

df_browse['mths_since_rcnt_il'] = df_browse['mths_since_rcnt_il'].fillna(median_rcnt_il_post)

In [234]:
# ============================================
# 4. Variables con bajo missing (<1%)
# ============================================

low_missing_cols = [
    'revol_util', 'dti', 'collections_12_mths_ex_med',
     'inq_last_6mths', 'open_acc',
    'earliest_cr_line', 'total_acc', 'pub_rec',
    'delinq_2yrs', 'acc_now_delinq'
]

df_browse = df_browse.dropna(subset=low_missing_cols)

In [235]:
# ============================================
# 5. Variables categóricas
# ============================================

df_browse['emp_length'] = df_browse['emp_length'].fillna('unknown')

In [236]:
# ============================================
# 6. Variables financieras (0 como valor válido)
# ============================================

financial_zero_impute = ['tot_cur_bal', 'tot_coll_amt']

for col in financial_zero_impute:

    df_browse[col + '_missing'] = df_browse[col].isna().astype(int)

    df_browse[col] = df_browse[col].fillna(0)

In [237]:
'''# ---------------------------------
# 1. Missingness flag
# ---------------------------------

df_browse['mths_since_last_delinq_missing'] = (
    df_browse['mths_since_last_delinq']
    .isna()
    .astype(int)
)

# ---------------------------------
# 2. Valor especial = max + 1
# ---------------------------------

max_delinq = df_browse['mths_since_last_delinq'].max()

df_browse['mths_since_last_delinq'] = (
    df_browse['mths_since_last_delinq']
    .fillna(max_delinq + 1)
)'''

"# ---------------------------------\n# 1. Missingness flag\n# ---------------------------------\n\ndf_browse['mths_since_last_delinq_missing'] = (\n    df_browse['mths_since_last_delinq']\n    .isna()\n    .astype(int)\n)\n\n# ---------------------------------\n# 2. Valor especial = max + 1\n# ---------------------------------\n\nmax_delinq = df_browse['mths_since_last_delinq'].max()\n\ndf_browse['mths_since_last_delinq'] = (\n    df_browse['mths_since_last_delinq']\n    .fillna(max_delinq + 1)\n)"

In [238]:
missing_table = pd.DataFrame({
    "Valores_Faltantes": df_browse.isna().sum(),
    "Porcentaje": df_browse.isna().mean() * 100,
})

missing_table = missing_table.sort_values(by="Porcentaje", ascending=False)
missing_table

,Valores_Faltantes,Porcentaje
mths_since_last_delinq,1156599,51.24395
open_acc_6m,0,0.00000
funded_amnt,0,0.00000
total_bal_il,0,0.00000
fico_range_low,0,0.00000
loan_amnt,0,0.00000
addr_state,0,0.00000
earliest_cr_line,0,0.00000
issue_d,0,0.00000
revol_util,0,0.00000


In [239]:
# ---------------------------------
# 1. Orden explícito para grade
# ---------------------------------

grade_order = ['A', 'B', 'C', 'D', 'E', 'F', 'G']

df_browse['grade'] = pd.Categorical(
    df_browse['grade'],
    categories=grade_order,
    ordered=True
)

In [240]:
# ---------------------------------
# 2. Orden explícito para sub_grade
# ---------------------------------

subgrade_order = [
    f'{grade}{n}'
    for grade in grade_order
    for n in range(1, 6)
]

df_browse['sub_grade'] = pd.Categorical(
    df_browse['sub_grade'],
    categories=subgrade_order,
    ordered=True
)

In [241]:
# ---------------------------------
# 3. Variable ordinal para emp_length
# ---------------------------------

emp_length_map = {
    'unknown': -1,
    '< 1 year': 0,
    '1 year': 1,
    '2 years': 2,
    '3 years': 3,
    '4 years': 4,
    '5 years': 5,
    '6 years': 6,
    '7 years': 7,
    '8 years': 8,
    '9 years': 9,
    '10+ years': 10
}

df_browse['emp_length_num'] = (
    df_browse['emp_length']
    .map(emp_length_map)
)

In [242]:
df_browse.shape

(2257045, 65)

In [248]:
df_browse.dtypes

open_acc_6m                          float64
total_bal_il                         float64
funded_amnt                          float64
earliest_cr_line              datetime64[ns]
fico_range_low                       float64
loan_amnt                            float64
addr_state                            object
tot_cur_bal                          float64
issue_d                       datetime64[ns]
revol_util                           float64
mths_since_rcnt_il                   float64
purpose                               object
grade                               category
tot_coll_amt                         float64
pub_rec                              float64
fico_range_high                      float64
annual_inc                           float64
inq_last_12m                         float64
inq_last_6mths                       float64
home_ownership                        object
mths_since_last_delinq               float64
revol_bal                            float64
il_util   

Realizamos una primer acercamiento al entendimiento y limpieza de los datos. La imputación de la variable `mths_since_last_delinq` será realizada despues del realizamiento del eda, que procederemos a hacer en el siguiente paso